# 🚀 GIAI ĐOẠN 3 — ĐỢT 2: HUẤN LUYỆN TOÀN DIỆN MÔ HÌNH STAIR-SRE-ANS v2
## Stepwise Spectral-Refined Contrastive Learning with Continuous Spectral Difficulty Decoupling & Decoupled Hardness-Aware Negative Scheduling (HANS)

---

### 📌 THÔNG TIN HỆ THỐNG & ĐỀ TÀI NGHIÊN CỨU
- **Đề tài**: Nâng cấp Mô hình Khuyến nghị Đa phương thức STAIR (Spectral-domain Multimodal Recommendation)
- **Cơ quan**: Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM (HCMUS) — Khóa Luận Tốt Nghiệp 2026
- **Kiến trúc đột phá**: **STAIR-SRE-ANS v2** (Phiên bản Đợt 2 Giai đoạn 3)
- **Môi trường thực thi**: Kaggle GPU Environment (Nvidia Tesla T4 16GB / P100)
- **Mục tiêu thực nghiệm**: Đột phá hiệu năng $\ge +5.0\%$ đồng thời trên cả 3 tập dữ liệu chuẩn Amazon Multimodal (Baby, Sports, Electronics), loại bỏ hoàn toàn các khiếm khuyết của đợt 1 (Evaluation Leak, Budget Bottleneck, Hard Dim-32 Boundary).

---

### 🛡️ TỔNG QUAN 5 TRỤ CỘT CÔNG NGHỆ CỐT LÕI (STAIR-SRE-ANS v2)

```
┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                 KIẾN TRÚC STAIR-SRE-ANS v2 TỔNG THỂ                                    │
└────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                        
  [Item Features] ──► [SVD Whitening (Baseline)] ──► [Pillar 1: Regularized Diagonal Spectral Projector]
                                                                        │ E_proj = E_svd ⊙ w
                                                                        ▼
                                                   [Forward Stepwise Convolution (FSC)]
                                                                        │
                                   ┌────────────────────────────────────┴───────────────────────────────────┐
                                   ▼                                                                        ▼
                   [Pairwise BPR Ranking Loss]                                         [Pillars 2-5: Stepwise SRE-ANS v2 Loss]
                                   │                                                                        │
                                   │                                              [Pillar 5: FIFO Memory Bank Queue (4096)]
                                   │                                                                        │
                                   │                                              [Pillar 2: Continuous Spectral Decoupling]
                                   │                                                  β(d) = 0.9·[1 - (d/64)^γ]
                                   │                                                  Separated L2 Norm: z_low, z_high
                                   │                                                  Difficulty D = α·S_low + (1-α)·S_high
                                   │                                                                        │
                                   │                                              [Pillar 4: Thresholded MFNA Gating]
                                   │                                                  W = σ(sim)·(1 + 0.5·M·max(0, cos))
                                   │                                                  attenuation = clamp(1 - W, 0, 1)
                                   │                                                                        │
                                   │                                              [Pillar 3: Gated Top-K Selection]
                                   │                                                  Score = Difficulty · Attenuation
                                   │                                                  k_hn Hard Negatives (Top-K)
                                   │                                                                        │
                                   │                                              [Stratified Penalty: Ψ_HN ≥ 1.0 > Ψ_EN]
                                   │                                                                        │
                                   │                                              [Vectorized InfoNCE with torch.gather]
                                   │                                                                        │
                                   │                                              [Pillar 5: Training-Guarded Enqueue]
                                   │                                                                        │
                                   └────────────────────────────────────┬───────────────────────────────────┘
                                                                        ▼
                                        L_total = L_BPR + λ_ans · L_ANS + λ_w · ||w - 1||_2^2
                                                                        │
                                                                        ▼
                                        [Decoupled HANS Scheduler: Monitor loss_ans at Epoch Boundary]
```

| STT | Trụ cột Công nghệ | Bản chất Toán học / Kỹ thuật | Khắc phục Triệt để Lỗi cũ |
|:---:|:---|:---|:---|
| **1** | **Regularized Diagonal Spectral Projector** | $E_{proj} = E_{svd} \odot w$, phạt neo giữ $\lambda_w \|w - 1\|_2^2$, $w$ cập nhật $0.1 \times \text{lr}$. | Bảo tồn tính trực giao SVD, triệt tiêu xoay không gian (0-rotation Jacobian). |
| **2** | **Continuous Spectral Difficulty Decoupling** | Đường cong phân rã liên tục $\beta(d) = 0.9 \cdot [1 - (d/D)^\gamma]$ trên cả 64 chiều, chuẩn hóa L2 riêng biệt $\mathbf{z}_{\text{low}}, \mathbf{z}_{\text{high}}$. | Xóa bỏ ranh giới cứng chiều 32 gượng ép; phản ánh 100% bản chất phổ tần của STAIR. |
| **3** | **Gated Top-K Selection** | Lọc False Negatives TRƯỚC khi gán ngân sách: $\text{Selection\_Score} = \mathcal{D} \times \text{attenuation}$. | Giải quyết triệt để Nghẽn Ngân sách (Budget Bottleneck), bảo toàn đủ số lượng Hard Negatives thực sự. |
| **4** | **Thresholded Cosine-Gated MFNA** | $W = \sigma(sim) \cdot [1 + 0.5 \cdot M_{meta} \cdot \max(0, \cos)]$, suy giảm an toàn $\text{clamp}(1 - W, 0, 1) \ge 0$. | Ngăn chặn suy giảm sai các item cùng category có khoảng cách góc lớn; bảo vệ thứ hạng nội ngành. |
| **5** | **Decoupled HANS & Training Guard** | Hàng đợi FIFO $Q \in \mathbb{R}^{4096 \times 64}$, chỉ enqueue khi `model.training`; HANS scheduler giám sát riêng biệt `loss_ans`. | Triệt tiêu hoàn toàn Rò rỉ Đánh giá (Evaluation Leak) và Bẫy Tự khuếch đại (Self-amplifying Loop). |

---

### 🎯 CHỈ TIÊU ĐỘT PHÁ CẦN ĐẠT (TARGET MILESTONES $\ge +5.0\%$)

| Dataset | Metric | STAIR Baseline | Phase 1 (SRE v1) | Phase 2 (v5 Best) | **Mục tiêu v2 ($\ge +5\%$)** |
|:---|:---:|:---:|:---:|:---:|:---:|
| **Amazon Baby** | Recall@20 / NDCG@20 | 0.1042 / 0.0454 | 0.0948 / 0.0412 | 0.1027 / 0.0454 | **$\ge 0.1095$ / $\ge 0.0480$** |
| **Amazon Sports** | Recall@20 / NDCG@20 | 0.1111 / 0.0500 | 0.1040 / 0.0466 | 0.1113 / 0.0508 | **$\ge 0.1168$ / $\ge 0.0530$** |
| **Amazon Electronics** | Recall@20 / NDCG@20 | 0.0665 / 0.0303 | 0.0601 / 0.0274 | 0.0700 / 0.0319 | **$\ge 0.0705$ / $\ge 0.0325$** |

## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã nguồn STAIR-SRE-ANS v2
Clone mã nguồn mới nhất từ GitHub branch `main`, kích hoạt môi trường CUDA, cài đặt dependencies và áp dụng compatibility patch cho `torchdata.datapipes` trên Kaggle.

In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced (v2)
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Luôn clone hoặc đồng bộ cưỡng bức repository mới nhất từ origin/main
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang đồng bộ cưỡng bức mã nguồn mới nhất...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
        print("✅ Đã reset về commit mới nhất của origin/main.")
    except Exception as e:
        print(f"Lỗi git fetch/reset ({e}), đang làm sạch và clone lại từ đầu...")
        shutil.rmtree(STAIR_DIR, ignore_errors=True)

if not os.path.exists(STAIR_DIR):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(STAIR_DIR)

# Xóa cache module để kernel luôn nạp phiên bản mới nhất từ đĩa
for mod_name in list(sys.modules.keys()):
    if 'stair_sre' in mod_name or 'models.stair_sre' in mod_name:
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc
print("Cài đặt dependencies (torchdata, freerec, torch-geometric, nvidia-ml-py, prettytable)...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml'
], check=True)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric',
    '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
], check=False)

# 3. Kaggle TorchData compatibility shims
import types
import torch.utils.data

try:
    import torchdata
    import torchdata.datapipes as dp
except Exception:
    dp = None

if dp is None or 'torchdata.datapipes' not in sys.modules:
    if 'torchdata' not in sys.modules:
        td = types.ModuleType('torchdata')
        sys.modules['torchdata'] = td
    else:
        td = sys.modules['torchdata']
    dp = types.ModuleType('torchdata.datapipes')
    td.datapipes = dp
    sys.modules['torchdata.datapipes'] = dp

if not hasattr(dp, 'iter'):
    iter_mod = types.ModuleType('torchdata.datapipes.iter')
    dp.iter = iter_mod
    sys.modules['torchdata.datapipes.iter'] = iter_mod
if not hasattr(dp.iter, 'IterDataPipe'):
    class IterDataPipe(torch.utils.data.IterableDataset):
        def __iter__(self): return iter([])
    dp.iter.IterDataPipe = IterDataPipe

if not hasattr(dp, 'map'):
    map_mod = types.ModuleType('torchdata.datapipes.map')
    dp.map = map_mod
    sys.modules['torchdata.datapipes.map'] = map_mod
if not hasattr(dp.map, 'MapDataPipe'):
    class MapDataPipe(torch.utils.data.Dataset):
        def __getitem__(self, idx): raise NotImplementedError
        def __len__(self): return 0
    dp.map.MapDataPipe = MapDataPipe

if not hasattr(dp, 'functional_datapipe'):
    def functional_datapipe(name, enable_df_datapipes_support=False):
        def decorator(cls):
            def method(self, *args, **kwargs):
                return cls(self, *args, **kwargs)
            if hasattr(dp, 'iter') and hasattr(dp.iter, 'IterDataPipe'):
                setattr(dp.iter.IterDataPipe, name, method)
            if hasattr(dp, 'map') and hasattr(dp.map, 'MapDataPipe'):
                setattr(dp.map.MapDataPipe, name, method)
            try:
                if hasattr(torch.utils.data, 'IterDataPipe'):
                    setattr(torch.utils.data.IterDataPipe, name, method)
                if hasattr(torch.utils.data, 'MapDataPipe'):
                    setattr(torch.utils.data.MapDataPipe, name, method)
            except Exception:
                pass
            return cls
        return decorator
    dp.functional_datapipe = functional_datapipe

# 4. Self-Healing Hotfix Guard: Đảm bảo models/stair_sre_ans_v2.py hỗ trợ tham số beta
ans_model_path = os.path.join(STAIR_DIR, 'models', 'stair_sre_ans_v2.py')
if os.path.exists(ans_model_path):
    with open(ans_model_path, 'r', encoding='utf-8') as f:
        src = f.read()
    if 'beta: Optional[torch.Tensor] = None' not in src:
        print("🔧 Áp dụng Self-Healing Hotfix cho models/stair_sre_ans_v2.py...")
        old_kw = 'subspace_alpha: float = 0.50,'
        new_kw = 'subspace_alpha: float = 0.50,\n        beta: Optional[torch.Tensor] = None,\n        gamma: float = 0.10,'
        if old_kw in src:
            src = src.replace(old_kw, new_kw)
        old_body = 'd_indices = torch.arange(dim, dtype=torch.float32)\n        beta_curve = 0.9 * (1.0 - torch.pow(d_indices / float(dim), 0.10))'
        new_body = '''if beta is not None:
            beta_curve = beta.detach().clone().to(dtype=torch.float32)
        else:
            d_indices = torch.arange(dim, dtype=torch.float32)
            beta_curve = 0.9 * (1.0 - torch.pow(d_indices / float(dim), float(gamma)))'''
        if old_body in src:
            src = src.replace(old_body, new_body)
        with open(ans_model_path, 'w', encoding='utf-8') as f:
            f.write(src)
        print("✅ Hotfix hoàn tất: StepwiseSREANSLoss đã hỗ trợ tham số beta!")

# 5. Kiểm tra GPU & Môi trường thực thi
print("=" * 70)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Phát hiện : {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print(f"✅ CUDA Version  : {torch.version.cuda}")
    print(f"✅ PyTorch Ver   : {torch.__version__}")
else:
    print("⚠️ CẢNH BÁO: Không phát hiện GPU CUDA! Vui lòng bật GPU Accelerator trên Kaggle.")
print("=" * 70)

## Cell 2 📂 Chuẩn bị Dữ liệu từ Kaggle Input (Tự động quét & Đồng bộ)
Quét toàn bộ `/kaggle/input` để phát hiện các thư mục dữ liệu `Amazon2014Baby_550_MMRec`, `Amazon2014Sports_550_MMRec`, `Amazon2014Electronics_550_MMRec` (hỗ trợ cả dạng nén zip/tar và lồng nhau) và đồng bộ vào `/kaggle/data`.

In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data
import os, shutil, glob

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

TARGET_DATASETS = {
    'baby':        'Amazon2014Baby_550_MMRec',
    'sports':      'Amazon2014Sports_550_MMRec',
    'electronics': 'Amazon2014Electronics_550_MMRec',
}

def scan_and_prepare_data():
    input_base = '/kaggle/input'
    found_datasets = {}
    print("🔍 Đang quét thư mục /kaggle/input để tìm dữ liệu...")
    
    for key, target_folder in TARGET_DATASETS.items():
        dst = os.path.join(DATA_ROOT, target_folder)
        if os.path.exists(dst) and len(os.listdir(dst)) >= 5:
            print(f"  [SẴN SÀNG] {target_folder} đã tồn tại trong {DATA_ROOT} ({len(os.listdir(dst))} tệp tin)")
            found_datasets[key] = dst
            continue

        candidates = []
        for root, dirs, files in os.walk(input_base):
            if target_folder in dirs:
                candidates.append(os.path.join(root, target_folder))
            has_modals = any('modality.pkl' in f for f in files)
            has_inter = any('train.csv' in f or 'train.tsv' in f or 'train.txt' in f for f in files)
            if has_modals and has_inter and (key in root.lower() or target_folder.lower() in root.lower()):
                candidates.append(root)

        if candidates:
            src = candidates[0]
            print(f"  [TÌM THẤY] {key} -> {src}")
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  [SAO CHÉP] Hoàn tất {key} sang {dst}")
            found_datasets[key] = dst
        else:
            archive_matches = glob.glob(f"{input_base}/**/*{key}*.*", recursive=True)
            archive_matches = [f for f in archive_matches if f.endswith(('.zip', '.tar.gz', '.tar', '.tgz'))]
            if archive_matches:
                arc = archive_matches[0]
                print(f"  [GIẢI NÉN] {arc} -> {dst}")
                os.makedirs(dst, exist_ok=True)
                if arc.endswith('.zip'):
                    import zipfile
                    with zipfile.ZipFile(arc, 'r') as zf:
                        zf.extractall(dst)
                elif arc.endswith(('.tar.gz', '.tar', '.tgz')):
                    import tarfile
                    with tarfile.open(arc, 'r:*') as tf:
                        tf.extractall(dst)
                found_datasets[key] = dst
            else:
                print(f"  [THIẾU] Chưa tìm thấy dữ liệu cho {target_folder}. Hãy đính kèm Kaggle Dataset!")

    return found_datasets

prepared_data = scan_and_prepare_data()
print("=" * 70)
print(f"Tổng số tập dữ liệu đã sẵn sàng: {len(prepared_data)} / {len(TARGET_DATASETS)}")
for k, v in prepared_data.items():
    print(f"  * {k.upper():12s}: {v}")
print("=" * 70)

## Cell 3 🧪 Kiểm tra Độc lập Module STAIR-SRE-ANS v2 (Bộ Unit Tests 5 Trụ cột Toán học)
Thực thi kiểm thử độc lập cho cả 5 trụ cột công nghệ của STAIR-SRE-ANS v2 trước khi chạy pipeline chính thức. Đảm bảo:
1. Trụ cột 1: Diagonal Projector $w$ có tính chất 0-rotation (Jacobi đường chéo) và tính phạt neo giữ $\lambda_w \|w-1\|^2$.
2. Trụ cột 2: Phân rã phổ liên tục $\beta(d)$ trên 64 chiều (không cắt cứng chiều 32), Separate L2 Norm cân bằng biên độ.
3. Trụ cột 3: Gated Top-K Selection giải phóng nghẽn ngân sách False Negatives.
4. Trụ cột 4: Thresholded Cosine-Gated MFNA bảo vệ phân loại nội ngành.
5. Trụ cột 5: Training Guard chặn 100% rò rỉ tập đánh giá vào FIFO Queue, và HANS scheduler phản ứng nhạy bén với plateau.
6. Gradient Flow: Lan truyền ngược (Backward pass) sạch 100%, không sinh ra NaN / Inf.

In [ ]:
# Cell 3: Kiểm tra STAIR-SRE-ANS v2 Module & Chạy Unit Tests 5 Trụ cột Toán học
import sys, os, inspect, torch
import torch.nn.functional as F

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(STAIR_DIR)

# Xóa cache module để đảm bảo nạp phiên bản mới nhất
for mod_name in list(sys.modules.keys()):
    if 'stair_sre' in mod_name or 'models.stair_sre' in mod_name:
        sys.modules.pop(mod_name, None)

from models.stair_sre_ans_v2 import (
    RegularizedDiagonalSpectralProjector,
    StepwiseSREANSLoss,
)

print('=' * 75)
print('BỘ KIỂM THỬ TOÀN DIỆN 5 TRỤ CỘT TOÁN HỌC: STAIR-SRE-ANS v2')
print('=' * 75)

dim = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------------------------------------------------
# TEST 1: Pillar 1 - Regularized Diagonal Spectral Projector (0-rotation)
# -------------------------------------------------------------------------
proj = RegularizedDiagonalSpectralProjector(dim=dim, reg_weight=1e-4).to(device)
assert torch.allclose(proj.w, torch.ones(dim, device=device)), "Projector w phải khởi tạo bằng 1.0!"
x = torch.randn(32, dim, device=device)
x_proj = proj(x)
assert torch.allclose(x, x_proj), "Tại Epoch 0 (w=1.0), proj(x) phải bằng x tuyệt đối (0-rotation)!"
assert proj.get_anchoring_loss().item() == 0.0, "L2 Anchoring loss tại w=1.0 phải bằng 0.0!"

# Thử nghiệm lệch trọng số và tính phạt
proj.w.data.add_(torch.randn(dim, device=device) * 0.05)
expected_anchoring = 1e-4 * torch.sum((proj.w - 1.0) ** 2)
assert torch.allclose(proj.get_anchoring_loss(), expected_anchoring), "L2 Anchoring loss tính sai công thức!"
print('  [PASS] Trụ cột 1: Diagonal Projector 0-rotation & L2 Anchoring Loss hoàn toàn chính xác.')

# -------------------------------------------------------------------------
# TEST 2: Pillar 2 - Continuous Spectral Difficulty Decoupling (No hard dim-32 jump)
# -------------------------------------------------------------------------
gamma = 0.2
beta3 = 0.1 + 0.9 * (torch.arange(dim, device=device, dtype=torch.float32) / dim).pow(gamma)
beta_fsc = 1.0 - beta3

sig = inspect.signature(StepwiseSREANSLoss.__init__)
loss_kwargs = {
    'dim': dim, 'tau': 0.20, 'queue_size': 512, 'warmup_epochs': 50,
    'gamma_max': 0.35, 'hn_ratio_max': 0.40, 'subspace_alpha': 0.50
}
if 'beta' in sig.parameters:
    loss_kwargs['beta'] = beta_fsc

sre_loss = StepwiseSREANSLoss(**loss_kwargs).to(device)

# Kiểm tra tính liên tục của hàm suy giảm phổ: không có điểm gãy tại chiều 32
diff_31_32 = abs(sre_loss.beta[31].item() - sre_loss.beta[32].item())
diff_0_1 = abs(sre_loss.beta[0].item() - sre_loss.beta[1].item())
print(f"         Độ dốc phổ: |β(31)-β(32)| = {diff_31_32:.6f} | |β(0)-β(1)| = {diff_0_1:.6f}")
assert diff_31_32 < 0.01, "Đường cong phổ không được có bước nhảy tại chiều 32!"
assert sre_loss.beta.shape[0] == dim and sre_loss.beta_high.shape[0] == dim
print('  [PASS] Trụ cột 2: Continuous Spectral Energy Decoupling liên tục 64 chiều (xóa bỏ cắt cứng 32).')

# -------------------------------------------------------------------------
# TEST 3: Pillar 4 - Thresholded Cosine-Gated False Negative Attenuation (MFNA)
# -------------------------------------------------------------------------
u_norm = F.normalize(torch.randn(16, dim, device=device), p=2, dim=-1)
neg_norm = F.normalize(torch.randn(512, dim, device=device), p=2, dim=-1)
cos_all = torch.matmul(u_norm, neg_norm.T)
sim_all = cos_all / 0.20

# Giả lập metadata mask (1: trùng category/brand)
meta_mask = (torch.rand(16, 512, device=device) > 0.8).float()
gated_meta = meta_mask * torch.clamp(cos_all, min=0.0)
W = torch.sigmoid(sim_all) * (1.0 + 0.5 * gated_meta)
attenuation = torch.clamp(1.0 - W, min=0.0, max=1.0)
assert torch.all(attenuation >= 0.0) and torch.all(attenuation <= 1.0), "Attenuation phải nằm trong đoạn [0, 1]!"
# Khi cos_all < 0, gated_meta = 0 dù meta_mask = 1
neg_sim_indices = (cos_all < 0) & (meta_mask == 1)
if neg_sim_indices.any():
    assert torch.all(gated_meta[neg_sim_indices] == 0.0), "Cosine âm phải chặn gated_metadata (bảo vệ intra-category)!"
print('  [PASS] Trụ cột 4: Thresholded Cosine-Gated MFNA bảo toàn thứ hạng nội ngành an toàn.')

# -------------------------------------------------------------------------
# TEST 4: Pillar 3 - Gated Top-K Selection (Loại bỏ Budget Bottleneck)
# -------------------------------------------------------------------------
difficulty = torch.rand(16, 512, device=device)
selection_score = difficulty * attenuation
k_hn = max(1, int(0.20 * 512))
_, hn_indices = torch.topk(selection_score, k=k_hn, dim=1)
assert hn_indices.shape == (16, k_hn), "Kích thước Top-K Hard Negatives phải đúng bằng [B, k_hn]!"
print('  [PASS] Trụ cột 3: Gated Top-K Selection lọc False Negatives trước khi phân bổ ngân sách HN.')

# -------------------------------------------------------------------------
# TEST 5: Pillar 5 - Training Guard on FIFO Queue (Zero Evaluation Leak)
# -------------------------------------------------------------------------
sre_loss.eval()
initial_ptr = int(sre_loss.queue_ptr.item())
dummy_pos = torch.randn(32, dim, device=device)
_ = sre_loss(torch.randn(32, dim, device=device), dummy_pos)
assert int(sre_loss.queue_ptr.item()) == initial_ptr, "Đang ở eval(), queue_ptr KHÔNG ĐƯỢC THAY ĐỔI (Tránh rò rỉ)!"

sre_loss.train()
_ = sre_loss(torch.randn(32, dim, device=device), dummy_pos)
assert int(sre_loss.queue_ptr.item()) == (initial_ptr + 32) % sre_loss.queue_size, "Trong train(), enqueue phải hoạt động!"
print('  [PASS] Trụ cột 5: Training Guard chặn 100% rò rỉ dữ liệu đánh giá vào FIFO Queue.')

# -------------------------------------------------------------------------
# TEST 6: Decoupled HANS Scheduler Dynamics
# -------------------------------------------------------------------------
sre_loss.warmup_epochs = 2
sre_loss.current_epoch = 0
sre_loss.update_scheduler(current_cl_loss=2.0)
assert sre_loss.gamma_h == 0.05 and sre_loss.hn_ratio == 0.10, "Warmup phase phải giữ tham số phạt cơ sở!"
# Vượt qua warmup
sre_loss.update_scheduler(current_cl_loss=1.8)
# Thêm lịch sử để kích hoạt trigger (plateau)
for _ in range(25):
    sre_loss.update_scheduler(current_cl_loss=1.80001, window=5, threshold=0.99)
assert sre_loss.gamma_h > 0.05, f"HANS phải nâng gamma_h khi loss plateau! Giá trị: {sre_loss.gamma_h}"
print(f'  [PASS] Trụ cột 5 (HANS): Scheduler phản ứng nhạy bén với plateau (gamma_h: {sre_loss.gamma_h:.4f}, hn_ratio: {sre_loss.hn_ratio:.4f}).')

# -------------------------------------------------------------------------
# TEST 7: End-to-End Backward Pass (Zero NaN / Inf)
# -------------------------------------------------------------------------
u_sample = torch.randn(16, dim, device=device, requires_grad=True)
i_sample = torch.randn(16, dim, device=device, requires_grad=True)
loss_val = sre_loss(u_sample, i_sample)
loss_val.backward()
assert u_sample.grad is not None and not torch.isnan(u_sample.grad).any(), "Gradient user chứa NaN!"
assert i_sample.grad is not None and not torch.isnan(i_sample.grad).any(), "Gradient item chứa NaN!"
print('  [PASS] End-to-End Backward: Gradient lan truyền mượt mà, 0% NaN / Inf.')

print('=' * 75)
print('🎉 XÁC NHẬN: TẤT CẢ 7 BÀI TEST TOÁN HỌC CỦA STAIR-SRE-ANS v2 ĐỀU ĐẠT CHUẨN SENIOR!')
print('=' * 75)

## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Trích xuất 4 Chỉ số
Thiết lập engine điều phối thực nghiệm:
- Giám sát luồng tiêu thụ VRAM nền qua `pynvml` (mẫu 2s/lần) để chứng minh tính ổn định bộ nhớ.
- Live stream output của `mainS3_v2.py` vào notebook đồng thời ghi log ra `/kaggle/working/logs/GD3/`.
- Regex parser tự động bóc tách đầy đủ 4 chỉ số khoa học: **Recall@10, Recall@20, NDCG@10, NDCG@20** tại Best Checkpoint và lịch sử tham số HANS.

In [ ]:
# Cell 4: Hàm hỗ trợ chạy Training v2 & Giám sát Phần cứng Toàn diện
import subprocess, threading, time, os, re, sys

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

BASELINE_REF = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_REF = {
    'baby':        {'Recall@10': 0.0611, 'Recall@20': 0.0948, 'NDCG@10': 0.0325, 'NDCG@20': 0.0412},
    'sports':      {'Recall@10': 0.0695, 'Recall@20': 0.1040, 'NDCG@10': 0.0376, 'NDCG@20': 0.0466},
    'electronics': {'Recall@10': 0.0401, 'Recall@20': 0.0601, 'NDCG@10': 0.0223, 'NDCG@20': 0.0274},
}

V1_1_REF = {
    'baby':        {'Recall@10': 0.0646, 'Recall@20': 0.1003, 'NDCG@10': 0.0341, 'NDCG@20': 0.0433},
    'sports':      {'Recall@10': 0.0723, 'Recall@20': 0.1098, 'NDCG@10': 0.0396, 'NDCG@20': 0.0493},
    'electronics': {'Recall@10': 0.0440, 'Recall@20': 0.0660, 'NDCG@10': 0.0244, 'NDCG@20': 0.0300},
}

V5_REF = {
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'electronics': {'Recall@10': 0.0465, 'Recall@20': 0.0700, 'NDCG@10': 0.0260, 'NDCG@20': 0.0319},
}

TARGET_G3_REF = {
    'baby':        {'Recall@10': 0.0710, 'Recall@20': 0.1095, 'NDCG@10': 0.0380, 'NDCG@20': 0.0480},
    'sports':      {'Recall@10': 0.0785, 'Recall@20': 0.1168, 'NDCG@10': 0.0430, 'NDCG@20': 0.0530},
    'electronics': {'Recall@10': 0.0470, 'Recall@20': 0.0705, 'NDCG@10': 0.0265, 'NDCG@20': 0.0325},
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def extract_best_test(log_path):
    # Trích xuất epoch tốt nhất và 4 chỉ số TEST (Recall@10, Recall@20, NDCG@10, NDCG@20)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        lines = content.splitlines()

    best_epoch = None
    best_metrics = {}

    ep_matches = re.findall(r'(?:Load best model @Epoch|TEST @Epoch:|Best @Epoch:?)\s*(\d+)', content, re.IGNORECASE)
    if ep_matches:
        best_epoch = int(ep_matches[-1])
    else:
        for line in reversed(lines):
            m = re.search(r'Epoch:\s*(\d+)', line)
            if m:
                best_epoch = int(m.group(1))
                break

    for line in reversed(lines):
        if 'TEST' in line and 'Avg:' in line:
            for metric in TRACKED_METRICS:
                m = re.search(rf'{metric}\s*Avg:\s*([0-9.]+)', line, re.IGNORECASE)
                if m:
                    best_metrics[metric] = float(m.group(1))
            if len(best_metrics) >= len(TRACKED_METRICS):
                break

    return best_epoch, best_metrics

def parse_training_loss(log_path):
    # Trích xuất hàm mất mát huấn luyện theo từng epoch
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    return [(int(ep), float(loss)) for ep, loss in matches]

def parse_valid_metric(log_path, metric='NDCG@20'):
    # Trích xuất chỉ số đánh giá tập validation theo từng epoch
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID @Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def parse_hans_trajectory(log_path):
    # Trích xuất quỹ đạo điều chỉnh siêu tham số HANS (gamma_h, hn_ratio, avg_cl_loss)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = r'\[HANS Epoch\s*(\d+)\]\s*gamma_h:\s*([0-9.]+)\s*\|\s*hn_ratio:\s*([0-9.]+)\s*\|\s*avg_cl_loss:\s*([0-9.]+)'
    matches = re.findall(pattern, content)
    return [(int(ep), float(gh), float(hnr), float(cl_loss)) for ep, gh, hnr, cl_loss in matches]

def run_training_sre_v2(key, yaml_cfg, data_root, log_path,
                        lambda_ans=5e-5, ans_tau=0.20, queue_size=4096,
                        warmup_epochs=50, gamma_max=0.35, hn_ratio_max=0.40,
                        subspace_alpha=0.50, reg_w=1e-4, lr_proj=None,
                        hans_window=10, ans_debug=False):
    # Runner chuẩn hóa thực thi huấn luyện STAIR-SRE-ANS v2 kèm theo dõi phần cứng
    print('=' * 80)
    print(f'🚀 BẮT ĐẦU HUẤN LUYỆN STAIR-SRE-ANS v2 (PHASE 3 / BATCH 2): {key.upper()}')
    print(f'  * Dataset Key         : {key}')
    print(f'  * YAML Config         : {yaml_cfg}')
    print(f'  * Log Path            : {log_path}')
    print(f'  * λ_ans (Weight)      : {lambda_ans}')
    print(f'  * τ (Temperature)     : {ans_tau}')
    print(f'  * Q (FIFO Queue Size) : {queue_size}')
    print(f'  * Warmup Epochs       : {warmup_epochs}')
    print(f'  * Ceiling (γ_h / hn)  : {gamma_max} / {hn_ratio_max}')
    print(f'  * Subspace Balance α  : {subspace_alpha}')
    print(f'  * Projector Anchor λ_w: {reg_w}')
    print(f'  * HANS Window Size    : {hans_window}')
    print('=' * 80)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    cmd = [
        sys.executable, '/kaggle/working/STAIR-Enhanced/mainS3_v2.py',
        '--config', yaml_cfg,
        '--root',   data_root,
        '--lambda-ans',     str(lambda_ans),
        '--ans-tau',        str(ans_tau),
        '--queue-size',     str(queue_size),
        '--warmup-epochs',  str(warmup_epochs),
        '--gamma-max',      str(gamma_max),
        '--hn-ratio-max',   str(hn_ratio_max),
        '--subspace-alpha', str(subspace_alpha),
        '--reg-w',          str(reg_w),
        '--hans-window',    str(hans_window),
    ]
    if lr_proj is not None:
        cmd.extend(['--lr-proj', str(lr_proj)])
    if ans_debug:
        cmd.append('--ans-debug')

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            f.write(line)
            f.flush()
        proc.wait()

    stop_evt.set()
    th.join(timeout=3.0)
    elapsed = time.time() - t0

    print('=' * 80)
    if proc.returncode != 0:
        print(f'❌ [THẤT BẠI] Quá trình huấn luyện {key.upper()} gặp lỗi (Exit Code: {proc.returncode})!')
    else:
        print(f'✅ [HOÀN TẤT] Huấn luyện {key.upper()} thành công trong {elapsed/60:.2f} phút ({elapsed:.1f}s)!')

    best_ep, metrics = extract_best_test(log_path)
    print(f'  * Checkpoint tối ưu : Epoch {best_ep}')
    for m, val in metrics.items():
        ref_bl = BASELINE_REF.get(key, {}).get(m, 0.0)
        gain = ((val - ref_bl) / ref_bl * 100) if ref_bl > 0 else 0.0
        sign = '+' if gain >= 0 else ''
        print(f'  * {m:12s}: {val:.4f} (So với Baseline: {sign}{gain:.2f}%)')

    if key in vram_profile and len(vram_profile[key]) > 0:
        peak = max(vram_profile[key])
        avg  = sum(vram_profile[key]) / len(vram_profile[key])
        print(f'  * VRAM Tiêu thụ     : Đỉnh = {peak:.1f} MB ({peak/1024:.2f} GB) | Trung bình = {avg:.1f} MB')
    print('=' * 80)

## Cell 5 📋 Cấu hình Siêu tham số STAIR-SRE-ANS v2 (Theo Thiết kế Chuẩn Báo cáo v2)
Thiết lập bộ siêu tham số chuẩn hóa cho 3 tập dữ liệu theo Mục 8.1 của Báo cáo STAIR3_v2_Report.md:
- **Amazon Baby**: $\lambda_{ans} = 5\times 10^{-5}$, $\tau = 0.20$, $Q = 4096$, $\text{warmup} = 50$, $\gamma_{max} = 0.35$, $hn\_ratio_{max} = 0.40$, $\text{window} = 10$.
- **Amazon Sports**: $\lambda_{ans} = 5\times 10^{-5}$, $\tau = 0.20$, $Q = 4096$, $\text{warmup} = 50$, $\gamma_{max} = 0.35$, $hn\_ratio_{max} = 0.40$, $\text{window} = 10$.
- **Amazon Electronics**: $\lambda_{ans} = 3\times 10^{-5}$, $\tau = 0.25$, $Q = 8192$, $\text{warmup} = 60$, $\gamma_{max} = 0.30$, $hn\_ratio_{max} = 0.35$, $\text{window} = 5$.

In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR-SRE-ANS v2 (Phase 3 / Batch 2)
import os

LOG_DIR_V2 = '/kaggle/working/logs/GD3'
os.makedirs(LOG_DIR_V2, exist_ok=True)

V2_CONFIGS = {
    'baby': {
        'yaml':           '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_550_MMRec.yaml',
        'log':            f'{LOG_DIR_V2}/baby3_v2.log',
        'lambda_ans':     5e-5,
        'ans_tau':        0.20,
        'queue_size':     4096,
        'warmup_epochs':  50,
        'gamma_max':      0.35,
        'hn_ratio_max':   0.40,
        'subspace_alpha': 0.50,
        'reg_w':          1e-4,
        'hans_window':    10,
    },
    'sports': {
        'yaml':           '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_550_MMRec.yaml',
        'log':            f'{LOG_DIR_V2}/sports3_v2.log',
        'lambda_ans':     5e-5,
        'ans_tau':        0.20,
        'queue_size':     4096,
        'warmup_epochs':  50,
        'gamma_max':      0.35,
        'hn_ratio_max':   0.40,
        'subspace_alpha': 0.50,
        'reg_w':          1e-4,
        'hans_window':    10,
    },
    'electronics': {
        'yaml':           '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_550_MMRec.yaml',
        'log':            f'{LOG_DIR_V2}/electronics3_v2.log',
        'lambda_ans':     3e-5,
        'ans_tau':        0.25,
        'queue_size':     8192,
        'warmup_epochs':  60,
        'gamma_max':      0.30,
        'hn_ratio_max':   0.35,
        'subspace_alpha': 0.50,
        'reg_w':          1e-4,
        'hans_window':    5,
    },
}

print("Bộ cấu hình siêu tham số STAIR-SRE-ANS v2 đã được thiết lập sẵn sàng!")
for ds, c in V2_CONFIGS.items():
    print(f"  * {ds.upper():12s}: λ_ans={c['lambda_ans']} | τ={c['ans_tau']} | Q={c['queue_size']} | Warmup={c['warmup_epochs']} | Window={c['hans_window']}")

## Cell 6 🏋️ Huấn luyện STAIR-SRE-ANS v2 trên Amazon Baby & Amazon Sports
Tiến hành huấn luyện 500 epochs trên 2 tập dữ liệu cốt lõi để đánh giá năng lực bứt phá hiệu năng của 5 trụ cột kiến trúc mới.

In [ ]:
# Cell 6: Huấn luyện STAIR-SRE-ANS v2 trên Baby & Sports
import torch

DATA_ROOT = '/kaggle/data'

# 1. Huấn luyện Amazon Baby
if 'baby' in prepared_data:
    cfg_b = V2_CONFIGS['baby']
    run_training_sre_v2(
        key            = 'baby',
        yaml_cfg       = cfg_b['yaml'],
        data_root      = DATA_ROOT,
        log_path       = cfg_b['log'],
        lambda_ans     = cfg_b['lambda_ans'],
        ans_tau        = cfg_b['ans_tau'],
        queue_size     = cfg_b['queue_size'],
        warmup_epochs  = cfg_b['warmup_epochs'],
        gamma_max      = cfg_b['gamma_max'],
        hn_ratio_max   = cfg_b['hn_ratio_max'],
        subspace_alpha = cfg_b['subspace_alpha'],
        reg_w          = cfg_b['reg_w'],
        hans_window    = cfg_b['hans_window'],
    )
else:
    print("⚠️ Bỏ qua Amazon Baby do chưa tìm thấy dữ liệu!")

# 2. Huấn luyện Amazon Sports
if 'sports' in prepared_data:
    cfg_s = V2_CONFIGS['sports']
    run_training_sre_v2(
        key            = 'sports',
        yaml_cfg       = cfg_s['yaml'],
        data_root      = DATA_ROOT,
        log_path       = cfg_s['log'],
        lambda_ans     = cfg_s['lambda_ans'],
        ans_tau        = cfg_s['ans_tau'],
        queue_size     = cfg_s['queue_size'],
        warmup_epochs  = cfg_s['warmup_epochs'],
        gamma_max      = cfg_s['gamma_max'],
        hn_ratio_max   = cfg_s['hn_ratio_max'],
        subspace_alpha = cfg_s['subspace_alpha'],
        reg_w          = cfg_s['reg_w'],
        hans_window    = cfg_s['hans_window'],
    )
else:
    print("⚠️ Bỏ qua Amazon Sports do chưa tìm thấy dữ liệu!")

## Cell 7 🚀 Huấn luyện STAIR-SRE-ANS v2 trên Amazon Electronics (~1.7M Tương tác)
Kiểm chứng khả năng mở rộng quy mô lớn (Scalability) và chứng minh bảo toàn bộ nhớ (Zero OOM) với hàng đợi FIFO mở rộng $Q = 8192$ và kích thước batch $4096$.

In [ ]:
# Cell 7: Huấn luyện STAIR-SRE-ANS v2 trên Amazon Electronics
import torch

DATA_ROOT = '/kaggle/data'

if 'electronics' in prepared_data:
    cfg_e = V2_CONFIGS['electronics']
    run_training_sre_v2(
        key            = 'electronics',
        yaml_cfg       = cfg_e['yaml'],
        data_root      = DATA_ROOT,
        log_path       = cfg_e['log'],
        lambda_ans     = cfg_e['lambda_ans'],
        ans_tau        = cfg_e['ans_tau'],
        queue_size     = cfg_e['queue_size'],
        warmup_epochs  = cfg_e['warmup_epochs'],
        gamma_max      = cfg_e['gamma_max'],
        hn_ratio_max   = cfg_e['hn_ratio_max'],
        subspace_alpha = cfg_e['subspace_alpha'],
        reg_w          = cfg_e['reg_w'],
        hans_window    = cfg_e['hans_window'],
    )
else:
    print("⚠️ Bỏ qua Amazon Electronics do chưa tìm thấy dữ liệu!")

## Cell 8 📊 Bảng So sánh Tổng hợp Ablation Study 8 Phiên bản (Đầy đủ 4 Chỉ số Khoa học)
Đối chiếu tiến trình phát triển hoàn chỉnh từ Baseline STAIR ban đầu, qua các mốc Giai đoạn 2 (v2a, v3, v4, v5), Giai đoạn 3 Đợt 1 (v1.1) tới **STAIR-SRE-ANS v2 (Đợt 2)** với đầy đủ 4 chỉ số chuẩn mực học thuật: **Recall@10, Recall@20, NDCG@10, NDCG@20**.

In [ ]:
# Cell 8: Bảng so sánh Ablation Study toàn diện 8 phiên bản (Recall@10, Recall@20, NDCG@10, NDCG@20)
import os
try:
    from prettytable import PrettyTable
    USE_PRETTYTABLE = True
except ImportError:
    USE_PRETTYTABLE = False

BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_RESULTS = {
    'baby':        {'Recall@10': 0.0611, 'Recall@20': 0.0948, 'NDCG@10': 0.0325, 'NDCG@20': 0.0412},
    'sports':      {'Recall@10': 0.0695, 'Recall@20': 0.1040, 'NDCG@10': 0.0376, 'NDCG@20': 0.0466},
    'electronics': {'Recall@10': 0.0401, 'Recall@20': 0.0601, 'NDCG@10': 0.0223, 'NDCG@20': 0.0274},
}

V2A_RESULTS = {
    'baby':        {'Recall@10': 0.0663, 'Recall@20': 0.1026, 'NDCG@10': 0.0351, 'NDCG@20': 0.0445},
    'sports':      {'Recall@10': 0.0738, 'Recall@20': 0.1102, 'NDCG@10': 0.0401, 'NDCG@20': 0.0494},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0658, 'NDCG@10': 0.0241, 'NDCG@20': 0.0298},
}

V3_RESULTS = {
    'baby':        {'Recall@10': 0.0680, 'Recall@20': 0.1050, 'NDCG@10': 0.0362, 'NDCG@20': 0.0458},
    'sports':      {'Recall@10': 0.0750, 'Recall@20': 0.1120, 'NDCG@10': 0.0410, 'NDCG@20': 0.0506},
    'electronics': {'Recall@10': 0.0445, 'Recall@20': 0.0670, 'NDCG@10': 0.0248, 'NDCG@20': 0.0306},
}

V4_RESULTS = {
    'baby':        {'Recall@10': 0.0666, 'Recall@20': 0.1037, 'NDCG@10': 0.0360, 'NDCG@20': 0.0453},
    'sports':      {'Recall@10': 0.0761, 'Recall@20': 0.1110, 'NDCG@10': 0.0417, 'NDCG@20': 0.0507},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0678, 'NDCG@10': 0.0259, 'NDCG@20': 0.0315},
}

V5_RESULTS = {
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'electronics': {'Recall@10': 0.0465, 'Recall@20': 0.0700, 'NDCG@10': 0.0260, 'NDCG@20': 0.0319},
}

V1_1_RESULTS = {
    'baby':        {'Recall@10': 0.0646, 'Recall@20': 0.1003, 'NDCG@10': 0.0341, 'NDCG@20': 0.0433},
    'sports':      {'Recall@10': 0.0723, 'Recall@20': 0.1098, 'NDCG@10': 0.0396, 'NDCG@20': 0.0493},
    'electronics': {'Recall@10': 0.0440, 'Recall@20': 0.0660, 'NDCG@10': 0.0244, 'NDCG@20': 0.0300},
}

TARGET_G3 = {
    'baby':        {'Recall@10': 0.0710, 'Recall@20': 0.1095, 'NDCG@10': 0.0380, 'NDCG@20': 0.0480},
    'sports':      {'Recall@10': 0.0785, 'Recall@20': 0.1168, 'NDCG@10': 0.0430, 'NDCG@20': 0.0530},
    'electronics': {'Recall@10': 0.0470, 'Recall@20': 0.0705, 'NDCG@10': 0.0265, 'NDCG@20': 0.0325},
}

v2_results = {}
for ds in ['baby', 'sports', 'electronics']:
    lp = f'{LOG_DIR_V2}/{ds}3_v2.log'
    ep, m = extract_best_test(lp)
    v2_results[ds] = {'epoch': ep, 'metrics': m}

METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

headers = [
    'Dataset', 'Chỉ số', 'Baseline', 'v5 (NE)', 'v1.1 (CNSS)',
    'v2 (SRE-ANS)', 'Mục tiêu G3', 'Δ vs BL (%)', 'Δ vs v5 (%)', 'Đạt ≥+5%?'
]
rows = []

for ds in ['baby', 'sports', 'electronics']:
    bl   = BASELINE[ds]
    v5   = V5_RESULTS[ds]
    v1_1 = V1_1_RESULTS[ds]
    v2   = v2_results[ds]['metrics']
    tgt  = TARGET_G3[ds]
    ep   = v2_results[ds]['epoch']

    for metric in METRICS:
        bl_v   = bl.get(metric, 0.0)
        v5_v   = v5.get(metric, 0.0)
        v1_1_v = v1_1.get(metric, 0.0)
        v2_v   = v2.get(metric, None)
        tgt_v  = tgt.get(metric, 0.0)

        if v2_v is not None:
            gain_bl = (v2_v - bl_v) / bl_v * 100.0
            gain_v5 = (v2_v - v5_v) / v5_v * 100.0
            v2_str  = f"{v2_v:.4f}"
            g_bl_s  = f"{gain_bl:+.2f}%"
            g_v5_s  = f"{gain_v5:+.2f}%"
            passed  = "✅ ĐẠT" if gain_bl >= 5.0 else ("⚠️ TIỆM CẬN" if gain_bl >= 2.0 else "❌")
        else:
            v2_str  = "Đang chạy..."
            g_bl_s  = "N/A"
            g_v5_s  = "N/A"
            passed  = "⏳"

        rows.append([
            f"{ds.upper()} (@Ep {ep})" if (metric == 'Recall@10' and ep) else ds.upper(),
            metric,
            f"{bl_v:.4f}",
            f"{v5_v:.4f}",
            f"{v1_1_v:.4f}",
            v2_str,
            f"{tgt_v:.4f}",
            g_bl_s,
            g_v5_s,
            passed
        ])

print("=" * 115)
print("BẢNG SO SÁNH HIỆU NĂNG TỔNG HỢP: STAIR-SRE-ANS v2 SO VỚI CÁC CỘT MỐC NGHIÊN CỨU")
print("=" * 115)

if USE_PRETTYTABLE:
    pt = PrettyTable()
    pt.field_names = headers
    for r in rows:
        pt.add_row(r)
    print(pt)
else:
    fmt = "{:<16} {:<12} {:<10} {:<10} {:<12} {:<14} {:<12} {:<14} {:<14} {:<10}"
    print(fmt.format(*headers))
    print("-" * 115)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
print("=" * 115)

## Cell 9 📈 Trực quan Hóa Quá trình Hội tụ & Động lực Học HANS (9 Đồ thị Chuyên nghiệp)
Vẽ hệ thống đồ thị khoa học phản ánh trực quan động lực học huấn luyện:
1. Hàng 1: Hàm mất mát huấn luyện (Training Loss Curves).
2. Hàng 2: Sự hội tụ của NDCG@20 và Recall@20 trên tập Validation qua 500 epochs.
3. Hàng 3: Quỹ đạo thích ứng tự động của siêu tham số HANS ($\gamma_h$ và $hn\_ratio$) phản ứng với plateau của InfoNCE loss.

In [ ]:
# Cell 9: Vẽ Biểu đồ Learning Curves & Quỹ đạo HANS Toàn diện
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
fig.suptitle('STAIR-SRE-ANS v2: Training Dynamics, Validation Convergence & HANS Trajectory', fontsize=16, fontweight='bold')

colors = {'baby': '#1f77b4', 'sports': '#2ca02c', 'electronics': '#ff7f0e'}

for col_idx, ds in enumerate(['baby', 'sports', 'electronics']):
    lp = f'{LOG_DIR_V2}/{ds}3_v2.log'
    c = colors[ds]
    
    # 1. Row 1: Training Loss Curve
    losses = parse_training_loss(lp)
    ax_loss = axes[0, col_idx]
    if losses:
        eps, l_vals = zip(*losses)
        ax_loss.plot(eps, l_vals, color=c, lw=1.8, label=f'{ds.upper()} Total Loss')
        ax_loss.set_title(f'Training Loss: {ds.upper()}', fontsize=12, fontweight='bold')
        ax_loss.set_xlabel('Epoch')
        ax_loss.set_ylabel('Loss')
        ax_loss.grid(True, alpha=0.3)
        ax_loss.legend()
    else:
        ax_loss.text(0.5, 0.5, 'Chưa có log / Đang chạy...', ha='center', va='center')
        ax_loss.set_title(f'Training Loss: {ds.upper()}')

    # 2. Row 2: Validation NDCG@20 & Recall@20 Convergence
    ndcg20_curve = parse_valid_metric(lp, 'NDCG@20')
    rec20_curve  = parse_valid_metric(lp, 'Recall@20')
    ax_val = axes[1, col_idx]
    if ndcg20_curve:
        e_n, v_n = zip(*ndcg20_curve)
        ax_val.plot(e_n, v_n, color='red', lw=1.8, label='NDCG@20')
        ax_val.axhline(y=BASELINE_REF[ds]['NDCG@20'], color='black', linestyle='--', alpha=0.7, label='Baseline')
        ax_val.axhline(y=TARGET_G3_REF[ds]['NDCG@20'], color='green', linestyle=':', lw=2, label='Target (+5%)')
        if rec20_curve:
            e_r, v_r = zip(*rec20_curve)
            ax_val2 = ax_val.twinx()
            ax_val2.plot(e_r, v_r, color='blue', lw=1.5, alpha=0.6, label='Recall@20 (Right)')
            ax_val2.set_ylabel('Recall@20', color='blue')
        ax_val.set_title(f'Validation Metrics: {ds.upper()}', fontsize=12, fontweight='bold')
        ax_val.set_xlabel('Epoch')
        ax_val.set_ylabel('NDCG@20', color='red')
        ax_val.grid(True, alpha=0.3)
        ax_val.legend(loc='lower right')
    else:
        ax_val.text(0.5, 0.5, 'Chưa có dữ liệu Validation', ha='center', va='center')
        ax_val.set_title(f'Validation: {ds.upper()}')

    # 3. Row 3: HANS Adaptive Trajectory (gamma_h & hn_ratio)
    hans_traj = parse_hans_trajectory(lp)
    ax_hans = axes[2, col_idx]
    if hans_traj:
        e_h, gh_vals, hnr_vals, cl_vals = zip(*hans_traj)
        ax_hans.plot(e_h, gh_vals, color='purple', lw=2.0, label='γ_h (HN Penalty)')
        ax_hans.plot(e_h, hnr_vals, color='orange', lw=2.0, label='hn_ratio (HN Pool %)')
        ax_hans.axvline(x=V2_CONFIGS[ds]['warmup_epochs'], color='gray', linestyle=':', label='Warmup End')
        ax_hans.set_title(f'HANS Scheduler: {ds.upper()}', fontsize=12, fontweight='bold')
        ax_hans.set_xlabel('Epoch')
        ax_hans.set_ylabel('Scheduler Coefficients')
        ax_hans.grid(True, alpha=0.3)
        ax_hans.legend()
    else:
        ax_hans.text(0.5, 0.5, 'Chưa có dữ liệu HANS', ha='center', va='center')
        ax_hans.set_title(f'HANS Trajectory: {ds.upper()}')

plt.tight_layout()
out_fig = '/kaggle/working/stair_sre_v2_convergence.png'
plt.savefig(out_fig, dpi=300)
print(f"✅ Đã xuất biểu đồ học thuật chất lượng cao tại: {out_fig}")
plt.show()

## Cell 10 ⚡ Biểu đồ Giám sát Bộ nhớ VRAM Thực tế (Chứng minh Zero OOM)
Đo lường mức tiêu thụ bộ nhớ GPU thực tế trong suốt quá trình chạy 500 epochs trên GPU Tesla T4 để chứng minh tính toán nhẹ và an toàn tuyệt đối của kiến trúc STAIR-SRE-ANS v2.

In [ ]:
# Cell 10: Vẽ Biểu đồ VRAM Profiling
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Hardware Telemetry: GPU VRAM Memory Profiling (Nvidia Tesla T4)', fontsize=14, fontweight='bold')

for idx, ds in enumerate(['baby', 'sports', 'electronics']):
    ax = axes[idx]
    records = vram_profile.get(ds, [])
    if records:
        times = [i * 2.0 / 60.0 for i in range(len(records))]
        ax.plot(times, records, color='#1f77b4', lw=2.0)
        peak = max(records)
        ax.axhline(y=peak, color='red', linestyle='--', label=f'Peak: {peak:.1f} MB ({peak/1024:.2f} GB)')
        ax.axhline(y=15000, color='gray', linestyle=':', label='T4 Cap: 15 GB')
        ax.set_title(f'{ds.upper()}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Thời gian (phút)')
        ax.set_ylabel('VRAM Sử dụng (MB)')
        ax.grid(True, alpha=0.3)
        ax.legend()
    else:
        ax.text(0.5, 0.5, f'Chưa có dữ liệu profiling {ds}', ha='center', va='center')
        ax.set_title(f'{ds.upper()}')

plt.tight_layout()
vram_fig = '/kaggle/working/stair_sre_v2_vram_profile.png'
plt.savefig(vram_fig, dpi=300)
print(f"✅ Đã xuất biểu đồ profiling bộ nhớ tại: {vram_fig}")
plt.show()

## Cell 11 💾 Xuất Báo Cáo Kết Quả CSV & Đoạn Mã LaTeX Cho Khóa Luận Tốt Nghiệp
Lưu trữ toàn bộ kết quả đối chuẩn khoa học ra tệp `/kaggle/working/ablation_phase3_stair_sre_v2_summary.csv` và sinh mã bảng biểu LaTeX sẵn sàng đưa vào Chương 4 & 5 của quyển Khóa Luận Tốt Nghiệp.

In [ ]:
# Cell 11: Xuất bảng kết quả CSV và mã LaTeX cho Khóa Luận Tốt Nghiệp
import csv

OUT_CSV = '/kaggle/working/ablation_phase3_stair_sre_v2_summary.csv'

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for r in rows:
        writer.writerow(r)

print(f"✅ Bảng kết quả tổng hợp đã được lưu trữ thành công tại: {OUT_CSV}")

# Sinh đoạn mã LaTeX chuẩn bị cho Khóa Luận
print("\n" + "=" * 80)
print("ĐOẠN MÃ BẢNG BIỂU LATEX (SẴN SÀNG CHO KHÓA LUẬN TỐT NGHIỆP):")
print("=" * 80)

latex_code = []
latex_code.append(r"\begin{table*}[htbp]")
latex_code.append(r"\centering")
latex_code.append(r"\caption{Bảng đối chuẩn hiệu năng STAIR-SRE-ANS v2 so với Baseline STAIR và các cải tiến qua các giai đoạn.}")
latex_code.append(r"\label{tab:stair_sre_v2_ablation}")
latex_code.append(r"\resizebox{\textwidth}{!}{")
latex_code.append(r"\begin{tabular}{llcccccccc}")
latex_code.append(r"\toprule")
latex_code.append(r"\textbf{Dataset} & \textbf{Chỉ số} & \textbf{Baseline} & \textbf{v5 (NE)} & \textbf{v1.1 (CNSS)} & \textbf{v2 (Ours)} & \textbf{Target G3} & \textbf{$\Delta$ vs BL (\%)} & \textbf{$\Delta$ vs v5 (\%)} & \textbf{Đạt $\ge +5\%$} \\")
latex_code.append(r"\midrule")

curr_ds = None
for r in rows:
    ds_name = r[0].split()[0]
    if ds_name != curr_ds:
        if curr_ds is not None:
            latex_code.append(r"\midrule")
        curr_ds = ds_name
    row_tex = " & ".join([str(x).replace("%", "\\%").replace("✅ ĐẠT", "\textbf{PASSED}").replace("⚠️ TIỆM CẬN", "SUB-TARGET").replace("❌", "FAILED").replace("⏳", "RUNNING") for x in r]) + r" \\"
    latex_code.append(row_tex)

latex_code.append(r"\bottomrule")
latex_code.append(r"\end{tabular}")
latex_code.append(r"}")
latex_code.append(r"\end{table*}")

tex_content = "\n".join(latex_code)
print(tex_content)

OUT_TEX = '/kaggle/working/stair_sre_v2_table.tex'
with open(OUT_TEX, 'w', encoding='utf-8') as f:
    f.write(tex_content)
print(f"\n✅ Đã lưu tệp LaTeX tại: {OUT_TEX}")

## 💡 Cẩm nang Vận hành & Phản biện Học thuật Trước Hội đồng (Field Guide)

### 1. Phân tích Động lực Học & Siêu tham số
- **Nhiệt độ InfoNCE $\tau$:**  
  Nếu nhận thấy `loss_ans` giảm quá nhanh về $0$ trong $30$ epoch đầu, hãy nâng $\tau$ từ $0.20$ lên $0.25$ (như đã áp dụng cho Electronics) để phân phối xác suất mềm hơn.
- **Hàng đợi FIFO Memory Bank $Q$:**  
  Với Electronics ($~1.7M$ tương tác), dung lượng $Q=8192$ là chìa khóa để bao phủ không gian mẫu âm mà chỉ tốn $\approx 2.1 \text{ MB}$ RAM, hoàn toàn không gây OOM.
- **Khung cửa sổ trượt HANS (Window size):**  
  Amazon Baby và Sports dùng $\text{window}=10$. Riêng Electronics dùng $\text{window}=5$ để phản ứng nhạy bén hơn với số batch lớn.

### 2. Kịch bản Phản biện Học thuật Mẫu mực Trước Hội đồng Khoa học
> **Câu hỏi của Hội đồng:** *"Tại sao mô hình không chia không gian biểu diễn thành 32 chiều Collaborative và 32 chiều Multimodal như cách tiếp cận phân vùng không gian cổ điển?"*
>
> **Kịch bản trả lời:**  
> *"Kính thưa Hội đồng Khoa học, việc coi vector 64 chiều của STAIR có 'ranh giới cứng tại chiều 32' là một giả thiết không phản ánh đúng bản chất giải tích vi phân của mạng lọc phổ STAIR:*
> 1. *Hàm phân rã năng lượng phổ của STAIR $\beta(d) = 0.9 \cdot [1 - (d/D)^\gamma]$ là một đường cong liên tục tuyệt đối trên toàn bộ 64 chiều. Chênh lệch giữa chiều 31 và 32 chỉ là $0.0026$ ($< 4.3\%$), trong khi toàn bộ sự suy giảm năng lượng đồ thị thực sự diễn ra ở 20 chiều đầu tiên.*
> 2. *Do đó, nhóm nghiên cứu đã xây dựng cơ chế **Continuous Spectral Difficulty Decoupling**: Sử dụng trực tiếp vector $\boldsymbol{\beta}$ liên tục và phổ bù $(\mathbf{1} - \boldsymbol{\beta})$ để chiết xuất thành phần Collaborative (tần số thấp) và Multimodal Invariant (tần số cao) trên toàn bộ 64 chiều, kết hợp chuẩn hóa L2 riêng biệt để loại bỏ hiện tượng thiên lệch biên độ.*
> 3. *Nhờ vậy, mô hình vừa bảo toàn 100% bản chất vật lý của STAIR, vừa giải quyết triệt để vấn đề ranh giới cơ học gượng ép."*